In [1]:
import torch

In [13]:
print("CUDA 사용 가능 여부:", torch.cuda.is_available())

CUDA 사용 가능 여부: True


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
pip install transformers[torch]

In [9]:
pip install librosa soundfile

Note: you may need to restart the kernel to use updated packages.


In [10]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
from datasets import load_dataset, Dataset, Audio
from tqdm import tqdm
import pandas as pd
from typing import Any, Dict, List

# 전처리된 데이터셋 로드 (예: CSV 파일로 저장한 경우)
data = pd.read_csv("whisper_finetuning_data_ds_5.csv")  # 'audio_path', 'transcription' 열 포함

# 데이터셋 생성 및 오디오 파일 로드 확인
dataset = Dataset.from_pandas(data)
dataset = dataset.cast_column("audio_path", Audio())  # 오디오 파일 로드

# Whisper 모델 및 프로세서 불러오기
model_name = "openai/whisper-medium"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name).to(device)

# 데이터셋 전처리 함수
def preprocess_data(examples):
    try:
        audio = examples["audio_path"]

        # 오디오 데이터가 올바르게 로드되지 않았을 경우 건너뛰기
        if not audio or "array" not in audio or "sampling_rate" not in audio:
            print(f"오디오 파일이 제대로 로드되지 않았습니다. 건너뛰기: {examples['audio_path']}")
            return {"input_features": None, "labels": None}

        inputs = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt").input_features
        labels = processor(text=examples["transcription"], return_tensors="pt").input_ids
        return {"input_features": inputs.squeeze(), "labels": labels.squeeze()}

    except Exception as e:
        print(f"오류 발생: {e}. 건너뛰기: {examples['audio_path']}")
        return {"input_features": None, "labels": None}

# tqdm을 사용해 데이터셋 전처리
print("데이터셋 전처리 중...")
dataset = dataset.map(preprocess_data, remove_columns=["audio_path", "transcription"])

# None 값이 포함된 데이터 필터링하여 제거
dataset = dataset.filter(lambda example: example["input_features"] is not None and example["labels"] is not None)


# 학습 파라미터 설정
training_args = TrainingArguments(
    output_dir="./whisper_finetuning",
    per_device_train_batch_size=4,
    evaluation_strategy="no", #epoch
    num_train_epochs=3,
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=100,
    learning_rate=5e-5
)

class WhisperDataCollator:
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        # input_features를 텐서로 변환
        input_features = [torch.tensor(feature["input_features"]) for feature in features]

        # input_features를 스택으로 변환
        input_features = torch.stack(input_features)

        # labels를 텐서로 변환하고 패딩
        labels = torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(feature["labels"]) for feature in features], 
            batch_first=True, 
            padding_value=-100
        )

        return {"input_features": input_features, "labels": labels}

        
# DataCollatorWithPadding 초기화
data_collator = WhisperDataCollator()

# Trainer 설정
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    eval_dataset=dataset,  # 여기서는 같은 데이터셋을 평가용으로 사용
    data_collator=data_collator
)

# 파인튜닝 진행 및 진행 상황 표시
print("Whisper 모델 파인튜닝 시작...")
trainer.train()  # trainer.train() 내에서 자체적으로 tqdm을 사용하여 진행 상황이 표시됩니다.

# 모델 저장
model.save_pretrained("./whisper_finetuned_model")
print("모델 파인튜닝 완료 및 저장 완료")

데이터셋 전처리 중...


Map:   0%|          | 0/418 [00:00<?, ? examples/s]

Filter:   0%|          | 0/418 [00:00<?, ? examples/s]

C:\Users\MATH-1\anaconda3\Lib\site-packages\transformers\training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Whisper 모델 파인튜닝 시작...


C:\Users\MATH-1\anaconda3\Lib\site-packages\transformers\models\whisper\modeling_whisper.py:545: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss


KeyboardInterrupt: 

In [17]:
# 모델 파라미터가 GPU에 있는지 확인
if torch.cuda.is_available():
    is_on_gpu = all(param.is_cuda for param in model.parameters())
    if is_on_gpu:
        print("모델이 GPU에 성공적으로 적용되었습니다.")
    else:
        print("모델이 GPU에 적용되지 않았습니다. CPU에서 실행 중입니다.")
else:
    print("GPU가 사용 가능하지 않습니다. CPU에서 실행 중입니다.")

모델이 GPU에 성공적으로 적용되었습니다.


In [ ]:
import zipfile
import os
import json
import pandas as pd
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Trainer, TrainingArguments
from datasets import Dataset, Audio
from tqdm import tqdm
import gc

# 데이터 폴더와 결과 저장 위치 설정
data_folder = 'C:/Users/MATH-1/dat_nlp/005.한영 혼합 인식 데이터/01.데이터/1.Training/라벨링데이터_0825_add'
wav_folder = 'C:/Users/MATH-1/dat_nlp/converted_wav_files'
output_data = []

# zip 파일당 최대 300개의 JSON 파일만 처리
for zip_file in sorted(os.listdir(data_folder)):
    zip_path = os.path.join(data_folder, zip_file)
    
    if zipfile.is_zipfile(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as z:
            json_count = 0
            for file in z.namelist():
                if file.endswith('.json') and json_count < 300:
                    with z.open(file) as f:
                        data = json.load(f)
                        dialogs = data.get("dialogs", [])
                        audio_file_name = os.path.basename(file).replace(".json", ".wav")
                        audio_path = os.path.join(wav_folder, audio_file_name)

                        for dialog in dialogs:
                            if "deleted" not in dialog:
                                text = dialog["text"]
                                for expression in dialog.get("expression", []):
                                    text = text.replace(expression["form"], expression["originalForm"])
                                output_data.append({
                                    "audio_path": audio_path,
                                    "transcription": text,
                                    "start_time": float(dialog["startTime"]),
                                    "end_time": float(dialog["endTime"])
                                })
                    json_count += 1

# DataFrame으로 정리 후 CSV로 저장
df = pd.DataFrame(output_data)
df.to_csv("whisper_finetuning_data.csv", index=False, encoding='utf-8-sig')

# 데이터셋 생성 및 전처리
data = pd.read_csv("whisper_finetuning_data.csv")
dataset = Dataset.from_pandas(data).cast_column("audio_path", Audio())

# Whisper 모델 및 프로세서 불러오기
model_name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name).to(device)

# 전처리 함수
def preprocess_data(examples):
    try:
        audio = examples["audio_path"]
        if not audio or "array" not in audio or "sampling_rate" not in audio:
            print(f"오디오 파일이 로드되지 않았습니다: {examples['audio_path']}")
            return {"input_features": None, "labels": None}
        
        inputs = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt").input_features
        labels = processor(text=examples["transcription"], return_tensors="pt").input_ids
        return {"input_features": inputs.squeeze(), "labels": labels.squeeze()}
    except Exception as e:
        print(f"오류 발생: {e}, 파일 건너뜀: {examples['audio_path']}")
        return {"input_features": None, "labels": None}

# 데이터셋 전처리 및 메모리 효율성 개선
print("데이터셋 전처리 중...")
dataset = dataset.map(preprocess_data, remove_columns=["audio_path", "transcription"], keep_in_memory=False, load_from_cache_file=False)
dataset = dataset.filter(
    lambda example: example["input_features"] is not None and example["labels"] is not None,
    writer_batch_size=8  # 메모리 효율 개선을 위해 설정
)

# 메모리 정리
gc.collect()

# 학습 파라미터 설정 (배치 크기 줄임)
training_args = TrainingArguments(
    output_dir="./whisper_finetuning",
    per_device_train_batch_size=2,
    evaluation_strategy="epoch",
    num_train_epochs=3,
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=100,
)

# Trainer 설정 및 학습 시작
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    eval_dataset=dataset,
)

print("Whisper 모델 파인튜닝 시작...")
trainer.train()
model.save_pretrained("./whisper_finetuned_model")
print("모델 파인튜닝 완료 및 저장 완료")
